In [1]:
#import libraries

import torch
import pandas as pd
import numpy as np 

from transformers import T5Tokenizer
from transformers import T5EncoderModel

In [ ]:
#load data
cedar = pd.read_csv("../../processed/combined_positives.csv")

#clean at dataset level

import re

def is_valid(seq):
    if not isinstance(seq, str):
        return False
    seq = seq.strip().upper()
    return bool(re.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", seq))

# Apply BOTH conditions together
data = cedar[
    cedar["mt_seq"].apply(is_valid) &
    cedar["wt_seq"].apply(is_valid)
].copy()

print("Before:", len(cedar))
print("After:", len(data))


Before: 11760
After: 11760


In [3]:
data.head() 

,mt_seq,wt_seq,label,data_source
0,ACDPHSGHFV,ARDPHSGHFV,1,CEDAR
1,AMLGTHTMEV,AMLSPHAMDV,1,CEDAR
2,EVDPIGHLY,EVVPISHLY,1,CEDAR
3,GVYDGEEHSV,GAYDGEEH,1,CEDAR
4,ITKKVADLVGF,KKVADLI,1,CEDAR


In [4]:
#convert into python lists

mt_seqs = data["mt_seq"].tolist()
wt_seqs = data["wt_seq"].tolist()
labels = data["label"].values 

In [6]:
#device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:
#load module
model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)

model = model.to(device)
model.eval() 

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


T5EncoderModel(
  (shared): Embedding(128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=4096, bias=False)
              (k): Linear(in_features=1024, out_features=4096, bias=False)
              (v): Linear(in_features=1024, out_features=4096, bias=False)
              (o): Linear(in_features=4096, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=16384, bias=False)
              (wo): Linear(in_features=16384, out_features=1024, bias=False)
              (dropout): Dropo

In [8]:
#embed function

def preprocess(seq): 
    return " ".join(list(seq))

def embed_seq(seq, batch_size=8):
    all_embeddings = []
    for i in range(0, len(seq), batch_size): 
        batch = seq[i:i+batch_size]
        #process seq
        batch = [preprocess(s) for s in batch]
        #convert to tokens
        inputs = tokenizer(batch, padding=True, return_tensors="pt")

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        #run model
        with torch.no_grad():
            outputs = model(input_ids = input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state 

        #mean pooling
        for j in range(len(batch)): 
            seq_len = len(batch[j].split()) #no of amino acids
            emb = hidden[j, :seq_len].mean(dim=0).cpu().numpy()
            all_embeddings.append(emb)
    return np.array(all_embeddings)

In [10]:
#run embeddings
mt_embeddings = embed_seq(mt_seqs)

In [11]:
wt_embeddings = embed_seq(wt_seqs)

In [ ]:
#save
np.savez("../../output/embeddings/prottrans_embeds_combined.npz", 
         mt_mean = mt_embeddings, 
         wt_mean = wt_embeddings, 
         labels=labels)